In [1]:
%run Hilbert_C^2_package.ipynb
import numpy as np
import resource
from sage.rings.polynomial.polydict import ETuple
#import psutil
#import os

In [2]:
def create_laurent_ring(var_specs, base_ring=QQ):
    """
    生成指定变量个数的多变量洛朗多项式环（支持负次数）
    var_specs: 列表，每个元素为(变量前缀, 个数)，如[('x',2), ('y',3)]
    base_ring: 基域，默认有理数域QQ
    """
    variables = []
    # 按输入生成变量名（如x0,x1,y0,y1,y2）
    for prefix, count in var_specs:
        variables.extend([f"{prefix}{i}" for i in range(count)])
    # 创建洛朗多项式环，变量顺序与生成顺序一致
    R = LaurentPolynomialRing(base_ring, variables)
    # 返回环和对应的变量（方便直接使用）
    return R

'''# ------------------- 使用示例 -------------------
# 输入1：x要2个（x0,x1），y要3个（y0,y1,y2）
var_input = [('x', 2), ('y', 3)]
# 生成环R和变量x0,x1,y0,y1,y2
R = create_laurent_ring(var_input)

# 验证：创建含负次数的多项式
f = 2*x0^(-1)*x1^2 + 3*y0*y1^(-3)*y2  # 包含x0^-1、y1^-3等负次数项
print("变量列表:", R.gens())  # 输出：('x0', 'x1', 'y0', 'y1', 'y2')
print("含负次数的多项式f:", f)  # 输出：2*x0^-1*x1^2 + 3*y0*y1^-3*y2
print(f.dict())'''

'# ------------------- 使用示例 -------------------\n# 输入1：x要2个（x0,x1），y要3个（y0,y1,y2）\nvar_input = [(\'x\', 2), (\'y\', 3)]\n# 生成环R和变量x0,x1,y0,y1,y2\nR = create_laurent_ring(var_input)\n\n# 验证：创建含负次数的多项式\nf = 2*x0^(-1)*x1^2 + 3*y0*y1^(-3)*y2  # 包含x0^-1、y1^-3等负次数项\nprint("变量列表:", R.gens())  # 输出：(\'x0\', \'x1\', \'y0\', \'y1\', \'y2\')\nprint("含负次数的多项式f:", f)  # 输出：2*x0^-1*x1^2 + 3*y0*y1^-3*y2\nprint(f.dict())'

### create_part2  definition
##### mapping is defined to be:
$$
k,r \mapsto \sum_{i=1}^k s_i z_i^r
$$

In [3]:
# convert mpoly.dict() back to a multivariable polynomial
# we didn't check whether the number of variables is aligned with which in the dictionary
# you need ensure this yourself
def load_dict(R, mpoly_dict):
    mpoly = sum([value * prod([R.gens()[i]^key[i] for i in range(len(key))])\
                for key, value in mpoly_dict.items()])
    return mpoly

# k, r --> \sum_1^k s_i*z_i^r
def create_part2(R, k, r):
    f = lambda i:tuple(1 if idx is i else 0 for idx in range(k)) + tuple(r if idx is i else 0 for idx in range(k))
    mpoly_dict = dict((f(idx), 1) for idx in range(k))
    res = load_dict(R, mpoly_dict)
    return res

# Return a polynomial with degree of s0,...,s_k-1 at most d whose j-th coefficients agree with self for all j<=d.
def truncate(f,k,R,d=2):
    truncated_dict = {key:value if all(deg <= d for deg in key[:k]) else 0 for key, value in f.dict().items()}
    res = load_dict(R, truncated_dict)
    return res

### create_fK_term2_part2  definition
$\lambda = (t2,t1)$ where $t1$ is $\lambda^+$, $t2$ is $\lambda^-$
##### mapping is defined to be:
$$
k,n,t1,t2,r_0 \mapsto \frac{r_0m_{-r_0}}{2}\left(\prod_{r\neq r_0}\left(\sum_{i=1}^k s_i z_i^r\right)^{m_{-r}}\right)\cdot \left(\sum_{i=1}^k s_i z_i^{r_0}\right)^{m_{-r_0}-1}\cdot \sum_{i=1}^k s_i^2z_i^{r_0}
$$

In [4]:
def create_fK_term2_part2(R,k,n,t1,t2,r0):
    # 计算需要统一的目标长度（取两者最大值）
    max_len = max(len(t1), len(t2))
    
    # 补0并转换为元组（切片保证不超出原长度，*[0]补够剩余长度）
    t1_padded = tuple(list(t1) + [0] * (max_len - len(t1)))
    t2_padded = tuple(list(t2) + [0] * (max_len - len(t2)))
    
    if r0 > 0:
        # define part 2 of expression of f_0,lambda^k
        fK_pos = (
            prod([create_part2(R, k, i+1)^t2_padded[i] if i != r0-1 else 1 for i in range(max_len)]) *
            create_part2(R, k, r0)^(t2_padded[r0-1]-1)
            * sum([R.gens()[idx]^2 * R.gens()[idx+k]^r0 for idx in range(k)]) * 1/2 * r0 * t2_padded[r0-1])
        #print(fK_pos)
        fK_neg = prod([create_part2(R, k, -i-1)^t1_padded[i] for i in range(max_len)])
        #print(fK_neg)
        fK_term1_part2 = fK_neg * fK_pos * R.gens()[k]^-n
        #print(fK_part2)
        return fK_term1_part2
    else:
        # define part 2 of expression of f_0,lambda^k
        fK_pos = prod([create_part2(R, k, i+1)^t2_padded[i] for i in range(max_len)])
        #print(fK_pos)
        fK_neg = (
            prod([create_part2(R, k, -i-1)^t1_padded[i] if i != -r0-1 else 1 for i in range(max_len)]) * 
            create_part2(R, k, r0)^(t1_padded[-r0-1]-1) * 
            sum([R.gens()[idx]^2 * R.gens()[idx+k]^r0 for idx in range(k)]) * 1/2 * r0 * t1_padded[-r0-1]
        )
        #print(fK_neg)
        fK_term1_part2 = truncate(fK_neg * fK_pos * R.gens()[k]^-n,k,R)
        #print(fK_part2)
        return fK_term1_part2

In [5]:
def get_constant_term(f1,f2):
    # Compute constant term of product of two multivariate Laurent polynomials
    res = 0
    for mono, coeff in f1.dict().items():
        if ETuple((-np.array(mono)).tolist()) in f2.dict():
            temp = coeff * f2.dict()[ETuple((-np.array(mono)).tolist())]
            res += temp
    return res

In [6]:
k = 4
n = 2
var_input = [('s', k), ('z', k)]
R = create_laurent_ring(var_input)

In [ ]:
# Part I: verify the result about f_0,lambda

In [19]:
l_pos = 1
l_neg = k + 1 - l_pos
s_pos = 4
s_neg = s_pos + n
P_pos_list = Partitions(s_pos, length = l_pos)
P_neg_list = Partitions(s_neg, length = l_neg)
lam_pos = P_pos_list.random_element()
lam_neg = P_neg_list.random_element()
print(lam_pos,lam_neg)

lam_pos_mul = partition_to_multiplicity(lam_pos)
lam_neg_mul = partition_to_multiplicity(lam_neg)
print(lam_neg_mul,lam_pos_mul)
#f = load_dict(R, {(-1, 2, 0, 0, 0): 2, (0, 0, 1, -3, 1): 3})
#print(f)
#f2 = create_part2(R, k, 3)
#print(f2)

# 统一两个tuple的长度，不足的部分用0补齐
# 定义两个原始元组
t1 = lam_pos_mul
t2 = lam_neg_mul

# 计算需要统一的目标长度（取两者最大值）
max_len = max(len(t1), len(t2))

# 补0并转换为元组（切片保证不超出原长度，*[0]补够剩余长度）
t1_padded = tuple(list(t1) + [0] * (max_len - len(t1)))
t2_padded = tuple(list(t2) + [0] * (max_len - len(t2)))
print(t1_padded,t2_padded )

# define part 2 of expression of f_0,lambda^k
f_pos = prod([create_part2(R, k, i+1)^t2_padded[i] for i in range(max_len)])
#print(f_pos)
f_neg = prod([create_part2(R, k, -i-1)^t1_padded[i] for i in range(max_len)])
#print(f_neg)
f_part2 = f_neg * f_pos * R.gens()[k]^-n
#print(f_part2)
#print(truncate(f_part2,k,R))

[4] [2, 2, 1, 1]
(2, 2) (0, 0, 0, 1)
(0, 0, 0, 1) (2, 2, 0, 0)


### temp1  definition
##### mapping is defined to be:
$$
j, maxlen \mapsto \sum_{i=1}^{j-1} s_i\left(\sum_{l=-maxlen}^{maxlen} lz_i^lz_j^{-l}\right)
$$

In [26]:
# define part 1 of expression of f_0,lambda^k
def temp1(j,max_len):
    # \sum_1^j-1 s_i * (\sum_(-maxlen)^maxlen l * z_i^l * z_j^-l)
    f = (
        lambda i:R.gens()[i]*
        sum([l * R.gens()[i+k]^l * R.gens()[j+k-1]^(-l)
             - l * R.gens()[i+k]^(-l) * R.gens()[j+k-1]^l for l in range(s_neg+1)])
    )
    res = sum([f(i) for i in range(j-1)])
    return res
'''j=3
test = temp1(j,max_len)
print(test)'''
f_part1 = prod([temp1(j,s_neg) for j in range(2, k+1)])
#print(f_part1)

In [27]:
f_0 = n * prod([R.gens()[idx]^-2 for idx in range(k)]) * f_part1 * f_part2
#print(f_0)
'''term_dict = {R.gens()[idx]:1 if idx < k else 0 for idx in range(2*k)}
print(term_dict)'''
res = f_0.constant_coefficient()
print('result is ',res)
print('should be ',factorial(k) * n^k)

result is  384
should be  384


In [ ]:
# Part II: Below we try to find pattern for f_K

In [18]:
l_pos = 2
l_neg = k - l_pos
s_pos = 5
s_neg = s_pos + n
P_pos_list = Partitions(s_pos, length = l_pos)
P_neg_list = Partitions(s_neg, length = l_neg)
lam_pos = P_pos_list.random_element()
lam_neg = P_neg_list.random_element()
print('test example is: ', lam_neg,lam_pos)

lam_pos_mul = partition_to_multiplicity(lam_pos)
lam_neg_mul = partition_to_multiplicity(lam_neg)
print(lam_neg_mul,lam_pos_mul)
#f = load_dict(R, {(-1, 2, 0, 0, 0): 2, (0, 0, 1, -3, 1): 3})
#print(f)
#f2 = create_part2(R, k, 3)
#print(f2)
# 统一两个tuple的长度，不足的部分用0补齐
# 定义两个原始元组
t1 = lam_pos_mul
t2 = lam_neg_mul

# 计算需要统一的目标长度（取两者最大值）
max_len = max(len(t1), len(t2))

# 补0并转换为元组（切片保证不超出原长度，*[0]补够剩余长度）
t1_padded = tuple(list(t1) + [0] * (max_len - len(t1)))
t2_padded = tuple(list(t2) + [0] * (max_len - len(t2)))

test example is:  [6, 1] [3, 2]
(1, 0, 0, 0, 0, 1) (0, 1, 1)


In [16]:
# define part 1 of expression of f_0,lambda^k
def temp1(j,max_len):
    # \sum_1^j-1 s_i * (\sum_(-maxlen)^maxlen l * z_i^l * z_j^-l)
    f = (
        lambda i:R.gens()[i]*
        sum([l * R.gens()[i+k]^l * R.gens()[j+k-1]^(-l)
             - l * R.gens()[i+k]^(-l) * R.gens()[j+k-1]^l for l in range(s_neg+1)])
    )
    res = sum([f(i) for i in range(j-1)])
    return res
'''j=3
test = temp1(j,max_len)
print(test)'''
fK_part1 = prod([temp1(j,max_len) for j in range(2, k+1)])

In [17]:
# define part 2 of expression of f_0,lambda^k
fK_pos = prod([create_part2(R, k, i+1)^t2_padded[i] for i in range(max_len)])
#print(fK_pos)
fK_neg = prod([create_part2(R, k, -i-1)^t1_padded[i] for i in range(max_len)])
#print(fK_neg)
fK_term1_part2 = truncate(fK_neg * fK_pos * R.gens()[k]^-n,k,R)
#print(fK_part2)

In [18]:
#fK_term1 = n * prod([R.gens()[idx]^-2 for idx in range(k)]) * fK_part1 * fK_term1_part2 * (-1/2 * n * R.gens()[0])
#print(f_0)
'''term_dict = {R.gens()[idx]:1 if idx < k else 0 for idx in range(2*k)}
print(term_dict)'''
'''res = (n * prod([R.gens()[idx]^-2 for idx in range(k)]) * fK_part1 * fK_term1_part2 * (-1/2 * n * R.gens()[0])
      ).constant_coefficient()'''
for idx in range(k):
    res = get_constant_term(
        n * prod([R.gens()[idx]^-2 for idx in range(k)]) * fK_part1 * (-1/2 * n * R.gens()[idx]),
        fK_term1_part2
    )
    print('multiply by ', R.gens()[idx])
    print('result is ',res*(-2))
    print('------')

multiply by  s0
result is  192
------
multiply by  s1
result is  192
------
multiply by  s2
result is  192
------
multiply by  s3
result is  192
------


In [11]:
for r in range(max_len):
    if t2_padded[r] != 0:
        r0 = r+1
        print(r0, t2_padded[r])
        #fK_term2_part2 = create_fK_term2_part2(R,k,n,t1,t2,r0)
        '''res = (
            n * prod([R.gens()[idx]^-2 for idx in range(k)]) * fK_part1 * fK_term2_part2
        ).constant_coefficient()'''
        
        res = get_constant_term(
            n * prod([R.gens()[idx]^-2 for idx in range(k)]) * fK_part1, create_fK_term2_part2(R,k,n,t1,t2,r0)
        )
        print('r = ',r0)
        print('result is ',res)
        print('normalized result is ', res/(r0 * t2_padded[r]))
        
        print('------------')

2 2
r =  2
result is  32
normalized result is  8
------------


In [12]:
for r in range(max_len):
    if t1_padded[r] != 0:
        r0 = -r-1
        print(r0, t1_padded[r])
        '''fK_term2 = (
            n * prod([R.gens()[idx]^-2 for idx in range(k)]) * fK_part1 * create_fK_term2_part2(R,k,n,t1,t2,r0)
        )'''
        res = get_constant_term(
            n * prod([R.gens()[idx]^-2 for idx in range(k)]) * fK_part1, create_fK_term2_part2(R,k,n,t1,t2,r0)
        )
        print('r = ',r0)
        print('result is ',res)
        print('normalized result is ', res/(r0 * t1_padded[r]))
        print('------------')

-2 1
r =  -2
result is  32
normalized result is  -16
------------


In [13]:
conf = lambda k,n,r:(-1)^k/2*factorial(k-2)*n*((r-n)^k + k*n^(k-1)*r - n^k)/r
for idx in [1,2,-3]:
    print(conf(6,1,idx))

60
72
-16308


In [ ]:
# compute term3


In [14]:
def fK(lam_neg,lam_pos):
    k = len(lam_neg) + len(lam_pos)
    n = sum(lam_neg) - sum(lam_pos)
    
    lam_pos_mul = partition_to_multiplicity(lam_pos)
    lam_neg_mul = partition_to_multiplicity(lam_neg)
    print(lam_neg_mul,lam_pos_mul)
    #f = load_dict(R, {(-1, 2, 0, 0, 0): 2, (0, 0, 1, -3, 1): 3})
    #print(f)
    #f2 = create_part2(R, k, 3)
    #print(f2)
    # 统一两个tuple的长度，不足的部分用0补齐
    # 定义两个原始元组
    t1 = lam_pos_mul
    t2 = lam_neg_mul

    # 计算需要统一的目标长度（取两者最大值）
    max_len = max(len(t1), len(t2))

    # 补0并转换为元组（切片保证不超出原长度，*[0]补够剩余长度）
    t1_padded = tuple(list(t1) + [0] * (max_len - len(t1)))
    t2_padded = tuple(list(t2) + [0] * (max_len - len(t2)))
    
    conf_term2 = lambda r:(-1)^k/2*factorial(k-2)*n*((r-n)^k + k*n^(k-1)*r - n^k)/r
    conf_term22 = lambda r:(-1)^k/2*factorial(k-2)*n*((r-n)^k)/r
    
    temp1 = [(r+1) * t2_padded[r] * conf_term2(r+1) if t2_padded[r] != 0 else 0 for r in range(max_len)]
    temp2 = [(-r-1) * t1_padded[r] * conf_term2(-r-1) if t1_padded[r] != 0 else 0 for r in range(max_len)]
    
    term2 = sum(temp1) + sum(temp2)
    
    res = -1/2*factorial(k-1)*n^(k+1) + term2
    return res

In [21]:
lam_neg = Partition([4])
lam_pos = Partition([3,2])
print(fK(lam_neg,lam_pos))

(0, 0, 0, 1) (0, 1, 1)
57


In [16]:
print((k-1)*(n+1)^k + (-k+1)^k, -(k-1)*n^k)

46 -16
